# PWV02 : Explore hologram data  quality
**Goal :** Study deviation with respect to night average

*Adapted to the 2026 merged files (runs 2023-2026) on 2026-11-13 : input file and options from `PWV00_parameters.py` (`atmfilename`, `version_run`, `tag`), columns `PWV [mm]_ram` (spectrogram) and `PWV [mm]_rum` (1D spectrum).*

- author Sylvie Dagoret-Campagne
- creation date 2026-01-30 : version run2026_v01
- last update : 2026-01-30
- last update : 2026-02-19 : runs from Corentin Ravoux : generate images with name having the run tag
- last update : 2026-03-02 : Correct for id and DATE-OBS mismatch on Corentin's runs
- last update : 2026-03-15 : Corentins concatenated files
- affiliation : IJCLab
- Kernel @usdf **w_2026_02**
- Home emac : base (conda)
- laptop : conda_py313

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from platform import python_version
print(python_version())

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
# must install the mysitcom package by doing at top level "pip install --user -e . "
from mysitcom.auxtel.pwv import scatter_datetime
from mysitcom.auxtel.pwv import strip_datetime
from mysitcom.auxtel.pwv import bar_counts_by_night
from mysitcom.auxtel.pwv import plot_dccd_chi2_vs_time
from mysitcom.auxtel.pwv import plot_dccd_chi2_vs_time_by_filter
from mysitcom.auxtel.pwv import stripplot_target_vs_time
from mysitcom.auxtel.pwv import plot_dccd_chi2_vs_time_by_target_filter
from mysitcom.auxtel.pwv import plot_dccd_chi2_histo_by_target_filter
from mysitcom.auxtel.pwv import plot_dccd_chi2_vs_time_by_target_filter_colorsedtype
from mysitcom.auxtel.pwv import plot_dccd_chi2_histo_by_target_filter_colorsedtype
from mysitcom.auxtel.pwv import summarize_dccd_chi2
from mysitcom.auxtel.pwv import plot_atmparam_vs_time, plot_atmparam_diff_vs_time
from mysitcom.auxtel.pwv import plot_atmparam_hist_per_filter, plot_atmparam_diff_hist_per_filter
from mysitcom.auxtel.pwv import GetNightMidnightsDict,GetNightBoundariesDict
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter,shiftaverage_column_data_byfilter

In [ ]:
import os

In [ ]:
# where are stored the figures
pathfigs = "figs_PWV02"
prefix = "pwv02"
if not os.path.exists(pathfigs):
    os.makedirs(pathfigs) 
figtype = ".png"

In [ ]:
import numpy as np
from numpy.linalg import inv
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm,SymLogNorm
from matplotlib.patches import Circle,Annulus
from astropy.visualization import ZScaleInterval
props = dict(boxstyle='round', facecolor="white", alpha=0.1)
#props = dict(boxstyle='round')

import matplotlib.colors as colors
import matplotlib.cm as cmx

import matplotlib.ticker                         # here's where the formatter is
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

from matplotlib.gridspec import GridSpec

from astropy.visualization import (MinMaxInterval, SqrtStretch,ZScaleInterval,PercentileInterval,
                                   ImageNormalize,imshow_norm)
from astropy.visualization.stretch import SinhStretch, LinearStretch,AsinhStretch,LogStretch

from astropy.io import fits
from astropy.wcs import WCS
from astropy import units as u
from astropy import constants as c

from scipy import interpolate
from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import KDTree, BallTree

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option('display.max_rows', 100)

import matplotlib.ticker                         # here's where the formatter is
import os
import re
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype

import pickle
from collections import OrderedDict

plt.rcParams["figure.figsize"] = (16,8)
plt.rcParams["axes.labelsize"] = 'xx-large'
plt.rcParams['axes.titlesize'] = 'xx-large'
plt.rcParams['xtick.labelsize']= 'xx-large'
plt.rcParams['ytick.labelsize']= 'xx-large'
plt.rcParams["legend.fontsize"] = "xx-large"

import scipy
from scipy.optimize import curve_fit,least_squares

from pprint import pprint

# new color correction model
import pickle
from scipy.interpolate import RegularGridInterpolator

In [ ]:
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

from astropy.visualization import (MinMaxInterval, SqrtStretch,ZScaleInterval,PercentileInterval,
                                   ImageNormalize,imshow_norm)
from astropy.visualization.stretch import SinhStretch, LinearStretch,AsinhStretch,LogStretch

from astropy.time import Time


In [ ]:
# Remove to run faster the notebook
#import ipywidgets as widgets
#%matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401
    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
from PWV00_parameters import *

In [ ]:
DumpConfig()

In [ ]:
from importlib.metadata import version

In [ ]:
# wavelength bin colors
#jet = plt.get_cmap('jet')
#cNorm = mpl.colors.Normalize(vmin=0, vmax=NSED)
#scalarMap = cmx.ScalarMappable(norm=cNorm, cmap=jet)
#all_colors = scalarMap.to_rgba(np.arange(NSED), alpha=1)

In [ ]:
np.__version__

In [ ]:
pd.__version__

### Configuration

In [ ]:
def convertNumToDatestr(num):
    year = num//10_000
    month= (num-year*10_000)//100
    day = (num-year*10_000-month*100)

    year_str = str(year).zfill(4)
    month_str = str(month).zfill(2)
    day_str = str(day).zfill(2)
    
    datestr = f"{year_str}-{month_str}-{day_str}"
    return pd.to_datetime(datestr)

In [ ]:
PWVMIN = 0.
PWVMAX = 20.

In [ ]:
FLAG_WITHCOLLIMATOR = False
DATE_WITHCOLLIMATOR = 20230930
datetime_WITHCOLLIMATOR = convertNumToDatestr(DATE_WITHCOLLIMATOR)
datetime_WITHCOLLIMATOR = pd.to_datetime("2023-09-30 00:00:00.0+0000")
datetime_WITHCOLLIMATOR

## Initialisation

### Read the file
- `atmfilename` is defined in `PWV00_parameters.py` 

In [ ]:
print(f"INPUT FILENAME : {atmfilename}")    

In [ ]:
the_suptitle = tag 

In [ ]:
inputfilename = atmfilename.split("/")[-1]

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec = pd.DataFrame(specdata)


In [ ]:
print(" | ".join(df_spec.columns)) 

In [ ]:
#df_spec.dtypes.to_frame('Type de donnée')

In [ ]:
# 1. Convert to datetime
#df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"])
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"],utc=True)
# 2. Sort by time (chronological order)
df_spec = df_spec.sort_values(by="Time", ascending=True).reset_index(drop=True)

In [ ]:
df_spec["nightObs"] = df_spec["id"] // 100_000

In [ ]:
df_spec["seq_num"]  = df_spec["id"] % 100_000

In [ ]:
df_spec["night_from_time_utc"] = (
    df_spec["Time"].dt.strftime("%Y%m%d").astype(int)
)

mismatch = df_spec[
    df_spec["night_from_time_utc"] != df_spec["nightObs"]
]

len(mismatch), len(df_spec)

In [ ]:
print(f"VERSION_RUN : {version_run}")

In [ ]:
FLAG_CORRECT_NIGHTOBS_VARIABLES = True

if FLAG_CORRECT_NIGHTOBS_VARIABLES:
 
    # Assurer UTC
    #df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)

    # Convertir au Chili
    df_spec["Time_local"] = df_spec["Time"].dt.tz_convert("America/Santiago")

    # Décalage de 12h vers le passé
    df_spec["nightObs_corr"] = (
        (df_spec["Time_local"] - pd.Timedelta(hours=12))
        .dt.strftime("%Y%m%d")
        .astype(int)
    )

In [ ]:
if FLAG_CORRECT_NIGHTOBS_VARIABLES:
    span = (
        df_spec.groupby("nightObs_corr")["Time"].max()
        - df_spec.groupby("nightObs_corr")["Time"].min()
    )

    print(span.max())

In [ ]:
if FLAG_CORRECT_NIGHTOBS_VARIABLES:
    df_spec["nightObs"] = df_spec["nightObs_corr"]
    

## Select only empty and OG550 filters

In [ ]:
df_spec["FILTER"].unique()

In [ ]:
if FLAG_PWVFILTERS: 
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST) ]

### Check Filters

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 8))

strip_datetime(
    df=df_spec,
    x="Time",
    y="FILTER",
    hue="FILTER",
    ax=ax,
    size=9,
)

plt.show()


In [ ]:
bar_counts_by_night(
    df=df_spec,
    night_col="nightObs",
    filter_col="FILTER",
    stacked=True,
    title="Observations per night (stacked)",
)

## Calculate midnights and night boundaries

In [ ]:
df_spec["nightObs"]

In [ ]:
DT = pd.Timedelta(minutes=7*24*60)
TMIN  = df_spec["Time"].min()-DT
TMAX  = df_spec["Time"].max()+DT

In [ ]:
# get night boundaries
dn = GetNightBoundariesDict(df_spec)
# get midnights
dnidnights = GetNightMidnightsDict(df_spec)

## Calculate PWV shift relative to its night average in Spectrogram

In [ ]:
df_spec = shiftaverage_column_data_byfilter(df_spec,"nightObs","FILTER","PWV [mm]_ram")

In [ ]:
df_spec.head()

### Plot PWV shift in spectrogram vs Time quality cuts

In [ ]:
fig,ax = plot_atmparam_vs_time(
    df_spec,
    time_col= "Time",
    filter_col = "FILTER",
    param_col = "PWV [mm]_ram_shift",
    param_err_col = "PWV [mm]_err_ram",
    title_param = "Shift PWV rel night average vs time (spectrogram) no cut qual. cut)",
    
    # seuils / bornes
    param_min_fig=-PWVMAX/5.,
    param_max_fig=PWVMAX/5.,
    param_min_cut=None,
    param_max_cut=None,
 
    # titres
    suptitle= the_suptitle,

    # axes externes
    axs=None,
    figsize=(18, 6),
)

if version_run not in ["run_v12"]:
    for key, tt in dn.items():
        ax.axvspan(tt[0],tt[1], color='blue', alpha=0.05,lw=0.5)

for key, midn in dnidnights.items():
    ax.axvline( midn ,color="purple",ls=":",alpha=0.5,lw=0.5)

plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plotvstime_pwvramrelnightaver_all-targets_allfilters-nocut"+figtype
plt.savefig(figname)
plt.show()

In [ ]:
fig,ax = plot_atmparam_hist_per_filter(
    df_spec,
    filter_col="FILTER",
    param_col = "PWV [mm]_ram_shift",
    param_range = (-PWVMAX/5.,PWVMAX/5.),

    # histogram control
    bins=100,
    density=True,
    hist_alpha=0.4,

    # x-axis limits
    param_min_fig=-PWVMAX/5,
    param_max_fig=PWVMAX/5.,

    title_param="Shift PWV ref night average vs time (spectrogram) no cut qual. cut)",
    # titres
    suptitle= the_suptitle
)

plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plothisto_pwvramrelnightaver_all-targets_allfilters-nocut"+figtype
plt.savefig(figname)
plt.show()

## Calculate PWV shift relative to its night average in Spectrum

In [ ]:
df_spec = shiftaverage_column_data_byfilter(df_spec,"nightObs","FILTER","PWV [mm]_rum")

## Calculate PWV shift relative to its night average in Spectrum

In [ ]:
fig,ax = plot_atmparam_vs_time(
    df_spec,
    time_col= "Time",
    filter_col = "FILTER",
    param_col = "PWV [mm]_rum_shift",
    param_err_col = "PWV [mm]_err_rum",
    title_param = "Shift PWV rel night average vs time (spectrum) no cut qual. cut)",
    
    # seuils / bornes
    param_min_fig=-PWVMAX/5.,
    param_max_fig=PWVMAX/5.,
    param_min_cut=None,
    param_max_cut=None,
 
    # titres
    suptitle= the_suptitle,

    # axes externes
    axs=None,
    figsize=(18, 6),
)

if version_run not in ["run_v12"]:
    for key, tt in dn.items():
        ax.axvspan(tt[0],tt[1], color='blue', alpha=0.05,lw=0.5)

for key, midn in dnidnights.items():
    ax.axvline( midn ,color="purple",ls=":",alpha=0.5,lw=0.5)

plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plotvstime_pwvrumrelnightaver_all-targets_allfilters-nocut"+figtype
plt.savefig(figname)
plt.show()

In [ ]:
fig,ax = plot_atmparam_hist_per_filter(
    df_spec,
    filter_col="FILTER",
    param_col = "PWV [mm]_rum_shift",
    param_range = (-PWVMAX/5.,PWVMAX/5.),

    # histogram control
    bins=100,
    density=True,
    hist_alpha=0.4,

    # x-axis limits
    param_min_fig=-PWVMAX/5,
    param_max_fig=PWVMAX/5.,

    title_param="Shift PWV ref night average vs time (spectrum) no cut qual. cut)",
    # titres
    suptitle= the_suptitle
)


plt.tight_layout()
figname =f"{pathfigs}/{prefix}_{version_run}_plothisto_pwvrumrelnightaver_all-targets_allfilters-nocut"+figtype
plt.savefig(figname)
plt.show()
